# Course figures — metamodel showcase

Export **slide-ready PNGs** for teaching / presentations.

**Story arc**
1. Domain — goals on the open boundary, WHACS hindcast links, buoy
2. Offshore forcing — bulk + partition Hs time series at the buoy's goal
3. LHS training — sampled SnapWave cases per goal
4. WHACS hindcast — observed bulk forcing at all goals (dynamic mode)
5. Buoy context — validation site on the domain map
6. Hero spectrum — buoy vs bulk vs partition reconstruct at storm peak
7. Pipeline schematic — workflow overview
8. Input / output — partition arrows at goals + reconstructed spectra at buoy

**Outputs:** `outputs/course_figures/*.png`

**Prerequisites:** goals GPKG, WHACS zarr, SnapWave LHS cases, fitted models,
reconstruction caches for the chosen `period` (notebooks 05–07, 10–11).



In [ ]:
from io import BytesIO
from pathlib import Path

import geopandas as gpd
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from PIL import Image
from utils.config import (
    PATHS,
    SIMULATIONS,
    goals_for_snapwave,
    load_goals,
    open_offshore_waves,
    resolve,
)
from utils.evaluation import validation as val
from utils.geometry.output_sites import compute_output_sites
from utils.model import metamodel as mm
from utils.observation import onshore_sites as obs
from utils.simulation.snapwave_wrappers import (
    build_enclosure_polygon,
    read_polygon_from_shp_file,
)
from utils.viz import case_plots as cp

## Configuration & helpers


In [ ]:
# --- slide / export settings ---
EXPORT_DIR = resolve("outputs/course_figures")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DPI = 200
FIGSIZE_WIDE = (13.33, 7.5)  # 16:9
FIGSIZE_TALL = (10.0, 11.0)

# --- case study ---
period = "2020-07"
storm_days = 5
forcing_moment = (
    None  # None → storm peak at buoy goal  # window around peak Hs for partition plot
)
offshore_name = "WHACS"
mda_n = 450
pca_variance = 0.99
wl = 0.0

buoy_instrument = "TODO_Buoy"  # instrument_id from your observations GeoJSON
buoy_site = obs.snapwave_site_id(buoy_instrument)
buoy_nc = "inputs/Observations/processed/TODO_buoy.nc"

recon_kw = dict(
    n=mda_n,
    offshore_name=offshore_name,
    goals=None,
    wl=wl,
    site_ids=[buoy_site],
    pca_variance=pca_variance,
)

GOALS_GPKG = resolve(SIMULATIONS["goals_file"]).with_suffix(".gpkg")
goals = load_goals()
goals_config = goals_for_snapwave(goals)
targets = gpd.read_file(GOALS_GPKG, layer="targets")
boundary_snaps_gdf = gpd.read_file(GOALS_GPKG, layer="boundary_snaps")
boundary_snaps = {
    str(int(row.goal_id)): {
        "longitude": float(row.geometry.x),
        "latitude": float(row.geometry.y),
    }
    for _, row in boundary_snaps_gdf.iterrows()
}

aoi_gdf, aoi_polygon = read_polygon_from_shp_file(resolve(PATHS["aoi_file"]))
enclosure_polygon = build_enclosure_polygon(
    aoi_polygon,
    float(SIMULATIONS.get("enclosure_buffer_m", 10_000)),
    n_vertices=4,
)
output_sites_df = compute_output_sites(aoi_polygon=aoi_polygon)

goal_map = val.closest_goals_table(
    [buoy_site], offshore_names=[offshore_name], goals=goals
)
buoy_goal = int(goal_map.goal.iloc[0])
print(f"Export dir : {EXPORT_DIR}")
print(f"Period     : {period}")
print(f"Buoy site  : {buoy_site}  →  closest goal G{buoy_goal}")


def apply_slide_style() -> None:
    plt.rcParams.update(
        {
            "font.size": 13,
            "axes.titlesize": 15,
            "axes.labelsize": 13,
            "legend.fontsize": 11,
            "figure.facecolor": "white",
            "savefig.facecolor": "white",
        }
    )


def save_fig(fig, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.png"
    fig.savefig(path, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"saved → {path}")
    return path


def build_lhs_cases_df() -> pd.DataFrame:
    """Forcing table from SnapWave LHS NetCDFs (one file per goal)."""
    chunks: list[pd.DataFrame] = []
    for gid in sorted(goals, key=int):
        with xr.open_dataset(mm.cases_path(gid)) as ds:
            df = (
                ds[["tp_forcing", "dir_forcing", "spr_forcing", "wl_forcing"]]
                .to_dataframe()
                .reset_index()
            )
        df["goal"] = int(gid)
        df["tp"] = df["tp_forcing"]
        df["dir"] = df["dir_forcing"]
        df["spr"] = df["spr_forcing"]
        df["wl"] = df["wl_forcing"]
        chunks.append(df[["goal", "tp", "dir", "spr", "wl"]])
    return pd.concat(chunks, ignore_index=True)


def pick_storm_peak_time(goal_id: int) -> pd.Timestamp:
    off = open_offshore_waves(offshore_name).sel(time=period, goal=goal_id)
    return pd.Timestamp(off["hs"].idxmax("time").values)


def storm_time_slice(peak: pd.Timestamp, days: int) -> slice:
    half = pd.Timedelta(days=days / 2)
    return slice(peak - half, peak + half)


def render_spec_image(
    da: xr.DataArray,
    title: str,
    *,
    figsize: tuple[float, float] = (5.5, 4.8),
    cmap: str = "magma",
) -> Image.Image:
    """Render one polar spectrum to a raster (wavespectra cannot facet on ax=)."""
    fig = plt.figure(figsize=figsize, facecolor="white")
    da.load().spec.plot(
        cmap=cmap,
        levels=32,
        normalised=True,
        add_colorbar=True,
        cbar_kwargs={"label": "E / Emax", "shrink": 0.78, "pad": 0.06},
    )
    ax = plt.gca()
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12, color="#1a1a2e")
    ax.set_facecolor("#fafbfc")
    ax.grid(True, alpha=0.25, linestyle=":", linewidth=0.6)
    buf = BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=DPI,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.1,
    )
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)


def stack_spec_images(images: list[Image.Image], *, pad: int = 12) -> Image.Image:
    """Vertically stack spectrum PNGs with white padding."""
    width = max(im.width for im in images)
    height = sum(im.height for im in images) + pad * (len(images) - 1)
    canvas = Image.new("RGB", (width, height), "white")
    y = 0
    for im in images:
        x = (width - im.width) // 2
        canvas.paste(im, (x, y))
        y += im.height + pad
    return canvas


apply_slide_style()

PARTITION_STYLE = {
    0: ("Wind sea (p0)", "#e41a1c"),
    1: ("Swell 1 (p1)", "#377eb8"),
    2: ("Swell 2 (p2)", "#4daf4a"),
    3: ("Swell 3 (p3)", "#984ea3"),
}


def resolve_forcing_moment(goal_id: int | None = None) -> pd.Timestamp:
    if forcing_moment is not None:
        return pd.Timestamp(forcing_moment)
    gid = buoy_goal if goal_id is None else goal_id
    return pick_storm_peak_time(gid)


def offshore_partitions_at(time: pd.Timestamp) -> xr.Dataset:
    return open_offshore_waves(offshore_name).sel(time=time, method="nearest")


def _partition_arrowprops(
    lon: float,
    lat: float,
    dir_deg: float,
    *,
    scale_deg: float,
    color: str,
    lw: float = 2.4,
    alpha: float = 0.92,
):
    """Arrow style with larger heads for course maps."""
    theta = np.deg2rad(float(dir_deg))
    dlon = np.sin(theta) * scale_deg
    dlat = np.cos(theta) * scale_deg
    return (
        dict(
            arrowstyle="-|>",
            color=color,
            lw=lw,
            alpha=alpha,
            shrinkA=0,
            shrinkB=2,
            mutation_scale=16,
        ),
        (lon + dlon, lat + dlat),
        (lon, lat),
    )


def course_map_extent(
    *, pad_lon: float = 0.15, pad_lat: float = 0.12
) -> tuple[float, float, float, float]:
    """Lon/lat limits that include AoI, all goals, WHACS nodes, and buoy."""
    lons: list[float] = []
    lats: list[float] = []
    for g in goals.values():
        lons.append(g["target"]["longitude"])
        lats.append(g["target"]["latitude"])
        hc = g["hindcast"]["whacs"]
        lons.append(hc["longitude"])
        lats.append(hc["latitude"])
    for snap in boundary_snaps.values():
        lons.append(snap["longitude"])
        lats.append(snap["latitude"])
    buoy = obs.get_coordinates()[buoy_instrument]
    lons.append(buoy["lon"])
    lats.append(buoy["lat"])
    bb = aoi_gdf.total_bounds
    lons.extend([float(bb[0]), float(bb[2])])
    lats.extend([float(bb[1]), float(bb[3])])
    return (
        min(lons) - pad_lon,
        max(lons) + pad_lon,
        min(lats) - pad_lat,
        max(lats) + pad_lat,
    )


def partition_legend_handles() -> list:
    """Legend entries with arrow glyphs for partition colours."""
    handles = []
    for part, (label, color) in sorted(PARTITION_STYLE.items()):
        handles.append(
            plt.Line2D(
                [0],
                [0],
                color=color,
                linewidth=3.2,
                marker=">",
                markersize=11,
                markeredgewidth=0,
                label=f"{label}  (dir = wave-from, len ∝ Hs)",
            )
        )
    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="black",
            markersize=7,
            linestyle="None",
            label="goal boundary snap",
        )
    )
    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor=cp.TYPE_COLORS["validation"],
            markersize=10,
            linestyle="None",
            label="buoy (spectra above)",
        )
    )
    return handles


def draw_goal_partition_arrows(
    ax,
    time: pd.Timestamp,
    *,
    arrow_base_deg: float = 0.30,
    hs_floor: float = 0.04,
    fan_offset_deg: float = 0.028,
) -> None:
    """Four partition arrows per goal (wave-from direction, length ∝ phs)."""
    off = offshore_partitions_at(time)
    goal_ids = sorted(int(g) for g in off.goal.values)
    part_vars = [f"phs{i}" for i in range(4)]
    hs_max = float(np.nanmax(off[part_vars].to_array("part").values))
    hs_max = max(hs_max, hs_floor)

    for gid in goal_ids:
        key = str(gid)
        snap = boundary_snaps[key]
        lon, lat = snap["longitude"], snap["latitude"]
        ax.plot(
            lon,
            lat,
            "o",
            color="white",
            markersize=7,
            markeredgecolor="black",
            markeredgewidth=1.0,
            zorder=6,
        )
        ax.annotate(
            f"G{gid}",
            (lon, lat),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
            fontweight="bold",
            color="#111111",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75),
            zorder=7,
        )

        for part in range(4):
            phs = float(off[f"phs{part}"].sel(goal=gid).values)
            pdir = float(off[f"pdir{part}"].sel(goal=gid).values)
            if not np.isfinite(phs) or phs < hs_floor or not np.isfinite(pdir):
                continue
            color = PARTITION_STYLE[part][1]
            scale = arrow_base_deg * (0.45 + 0.55 * phs / hs_max)
            theta = np.deg2rad(pdir)
            perp = theta + np.pi / 2
            shift = (part - 1.5) * fan_offset_deg
            lon0 = lon + shift * np.sin(perp)
            lat0 = lat + shift * np.cos(perp)
            props, xytext, xy = _partition_arrowprops(
                lon0,
                lat0,
                pdir,
                scale_deg=scale,
                color=color,
            )
            ax.annotate("", xy=xy, xytext=xytext, arrowprops=props, zorder=5)


def stack_spec_images_h(images: list[Image.Image], *, pad: int = 14) -> Image.Image:
    width = sum(im.width for im in images) + pad * (len(images) - 1)
    height = max(im.height for im in images)
    canvas = Image.new("RGB", (width, height), "white")
    x = 0
    for im in images:
        y = (height - im.height) // 2
        canvas.paste(im, (x, y))
        x += im.width + pad
    return canvas

## Fig 1 — Domain map

Goals on the AoI boundary (propagation wedges), dashed links to **WHACS** hindcast
nodes, and the validation buoy inside the domain.


In [ ]:
apply_slide_style()
goal_ids = sorted(int(k) for k in goals)
colors = cp._goal_colors(goal_ids)
buoy_meta = obs.get_coordinates()[buoy_instrument]

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
cp.plot_base_map(ax=ax, title="Offshore goals & buoy", show_topobathy=True)
cp.plot_sectors(targets, goals, boundary_snaps, ax, radius_deg=0.28)

for gid in goal_ids:
    key = str(gid)
    snap = boundary_snaps[key]
    hc = goals[key]["hindcast"]["whacs"]
    color = colors[gid]
    ax.plot(
        [snap["longitude"], hc["longitude"]],
        [snap["latitude"], hc["latitude"]],
        color=color,
        linewidth=1.2,
        alpha=0.75,
        zorder=5,
    )
    ax.plot(
        hc["longitude"],
        hc["latitude"],
        marker="o",
        color=color,
        markersize=5,
        markeredgecolor="white",
        markeredgewidth=0.4,
        zorder=6,
    )

ax.plot(
    buoy_meta["lon"],
    buoy_meta["lat"],
    marker="s",
    color=cp.TYPE_COLORS["validation"],
    markersize=12,
    markeredgecolor="white",
    markeredgewidth=0.8,
    zorder=8,
    label="validation buoy",
)
ax.annotate(
    "buoy",
    (buoy_meta["lon"], buoy_meta["lat"]),
    xytext=(6, 6),
    textcoords="offset points",
    fontsize=11,
    color=cp.TYPE_COLORS["validation"],
    weight="bold",
)

handles = [
    mpatches.Patch(facecolor=colors[g], alpha=0.5, label=f"Goal {g}")
    for g in goal_ids[:4]
]
handles.append(
    plt.Line2D([0], [0], color="gray", linestyle="--", label="target → boundary")
)
handles.append(
    plt.Line2D([0], [0], color="gray", linewidth=1.2, label="boundary → WHACS")
)
handles.append(
    plt.Line2D(
        [0],
        [0],
        marker="s",
        color="w",
        markerfacecolor=cp.TYPE_COLORS["validation"],
        markersize=10,
        label="buoy",
    )
)
ax.legend(handles=handles, loc="lower left", fontsize=9, framealpha=0.92)
x0, x1, y0, y1 = course_map_extent()
ax.set_xlim(x0, x1)
ax.set_ylim(y0, y1)

save_fig(fig, "fig01_domain")

## Fig 2 — Offshore partitions (time series)

WHACS **bulk Hs** and partition components ``phs0``–``phs3`` at the goal closest
to the buoy, centred on the storm peak in ``period``.


In [ ]:
apply_slide_style()
peak_time = pick_storm_peak_time(buoy_goal)
t_slice = storm_time_slice(peak_time, storm_days)

off = open_offshore_waves(offshore_name).sel(time=t_slice, goal=buoy_goal).load()
times = pd.to_datetime(off.time.values)

partition_style = {
    "phs0": ("Wind sea (p0)", "#e41a1c"),
    "phs1": ("Swell 1 (p1)", "#377eb8"),
    "phs2": ("Swell 2 (p2)", "#4daf4a"),
    "phs3": ("Swell 3 (p3)", "#984ea3"),
}

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
for var, (label, color) in partition_style.items():
    if var in off:
        ax.plot(times, off[var].values, label=label, color=color, linewidth=1.6)

ax.plot(
    times,
    off["hs"].values,
    label="Bulk Hs (p−1)",
    color="black",
    linewidth=2.2,
    linestyle="--",
)
ax.axvline(peak_time, color="gray", linestyle=":", linewidth=1.2, label="storm peak")
ax.set_title(
    f"WHACS offshore Hs — goal G{buoy_goal} — {period}\n"
    f"peak @ {peak_time:%Y-%m-%d %H:%M} UTC"
)
ax.set_xlabel("Time")
ax.set_ylabel("Hs (m)")
ax.legend(loc="upper right", ncol=2)
ax.grid(True, alpha=0.25)

save_fig(fig, "fig02_offshore_partitions_ts")

## Fig 3 — Metamodel training (LHS cases)

Sampled wave directions per goal from the **500 LHS SnapWave cases** (notebook 05).
Reuses the same diagnostic layout as case generation.


In [ ]:
apply_slide_style()
lhs_cases = build_lhs_cases_df()
print(
    f"LHS cases loaded: {len(lhs_cases):,} rows across {lhs_cases['goal'].nunique()} goals"
)

fig = cp.plot_metamodel_configuration(
    aoi_gdf=aoi_gdf,
    enclosure_polygon=enclosure_polygon,
    goals_config=goals_config,
    cases_df=lhs_cases,
    output_sites_df=output_sites_df,
    bounds=SIMULATIONS.get("bounds"),
    num_samples_per_goal=SIMULATIONS.get("num_samples_per_goal"),
    mesh_file=PATHS.get("mesh_file"),
    enclosure_buffer_m=float(SIMULATIONS.get("enclosure_buffer_m", 10_000)),
    sector_radius_deg=0.30,
    arrow_scale_deg=0.12,
    max_arrows_per_goal=18,
    show_topobathy=True,
    title="Metamodel training — LHS cases per goal",
)
save_fig(fig, "fig03_metamodel_lhs")

## Fig 4 — WHACS hindcast forcing (dynamic mode)

Observed bulk forcing from WHACS at every goal for ``period`` — the input that
drives **dynamic** SnapWave runs and **reconstruction** hindcasts.


In [ ]:
apply_slide_style()
offshore = open_offshore_waves(offshore_name).sel(time=period)

fig = cp.plot_dynamic_configuration(
    aoi_gdf=aoi_gdf,
    enclosure_polygon=enclosure_polygon,
    goals_config=goals_config,
    offshore=offshore,
    output_sites_df=output_sites_df,
    offshore_name=offshore_name,
    mesh_file=PATHS.get("mesh_file"),
    enclosure_buffer_m=float(SIMULATIONS.get("enclosure_buffer_m", 10_000)),
    sector_radius_deg=0.30,
    arrow_scale_deg=0.12,
    max_arrows_per_goal=18,
    max_dir_marks=150,
    show_topobathy=True,
    title=f"WHACS hindcast forcing — {period}",
)
save_fig(fig, "fig04_whacs_hindcast_forcing")

## Fig 5 — Buoy on domain (validation context)

Zoomed map highlighting the buoy, its closest goal, and the WHACS node used for
reconstruction at that goal.


In [ ]:
apply_slide_style()
buoy_meta = obs.get_coordinates()[buoy_instrument]
row = goal_map.iloc[0]
gkey = str(buoy_goal)
snap = boundary_snaps[gkey]
hc = goals[gkey]["hindcast"]["whacs"]
tgt = goals[gkey]["target"]

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
cp.plot_base_map(ax=ax, title="Buoy validation site", show_topobathy=True)

left, right = goals[gkey]["angles"]
theta1, theta2 = cp.nautical_sector_to_wedge_angles(left, right)
ax.add_patch(
    mpatches.Wedge(
        (snap["longitude"], snap["latitude"]),
        0.18,
        theta1,
        theta2,
        facecolor="#1f78b4",
        alpha=0.25,
        edgecolor="#1f78b4",
        linewidth=1.0,
    )
)
ax.plot(
    [tgt["longitude"], snap["longitude"]],
    [tgt["latitude"], snap["latitude"]],
    "k--",
    alpha=0.5,
    linewidth=0.9,
)
ax.plot(
    [snap["longitude"], hc["longitude"]],
    [snap["latitude"], hc["latitude"]],
    color="#1f78b4",
    linewidth=1.8,
)
ax.plot(
    hc["longitude"],
    hc["latitude"],
    "o",
    color="#1f78b4",
    markersize=9,
    markeredgecolor="white",
)
ax.plot(snap["longitude"], snap["latitude"], "kx", markersize=7)
ax.plot(
    buoy_meta["lon"],
    buoy_meta["lat"],
    "s",
    color=cp.TYPE_COLORS["validation"],
    markersize=14,
    markeredgecolor="white",
    markeredgewidth=0.8,
    zorder=8,
)
ax.annotate(
    f"G{buoy_goal} WHACS",
    (hc["longitude"], hc["latitude"]),
    xytext=(8, -10),
    textcoords="offset points",
    fontsize=10,
)
ax.annotate(
    "buoy",
    (buoy_meta["lon"], buoy_meta["lat"]),
    xytext=(8, 6),
    textcoords="offset points",
    fontsize=11,
    color=cp.TYPE_COLORS["validation"],
    weight="bold",
)

pad = 0.35
ax.set_xlim(
    min(buoy_meta["lon"], hc["longitude"]) - pad,
    max(buoy_meta["lon"], hc["longitude"]) + pad,
)
ax.set_ylim(
    min(buoy_meta["lat"], hc["latitude"]) - pad,
    max(buoy_meta["lat"], hc["latitude"]) + pad,
)
ax.set_aspect("equal", adjustable="box")
ax.text(
    0.02,
    0.02,
    f"closest goal: G{buoy_goal}  ({row.distance_m / 1000:.1f} km to WHACS node)",
    transform=ax.transAxes,
    fontsize=10,
    va="bottom",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
)

save_fig(fig, "fig05_buoy_context_map")

## Fig 6 — Hero spectrum (storm peak)

Polar spectra at the buoy for the **storm-peak hour**: observed buoy, bulk
reconstruct (partition −1), and partition-sum reconstruct (0–3).

Requires v5 reconstruction spectra caches for ``period``.


In [ ]:
apply_slide_style()
peak_time = pick_storm_peak_time(buoy_goal)
print(f"Hero spectrum time: {peak_time}")

buoy_raw = xr.open_dataset(resolve(buoy_nc))
spec_var = "Efth" if "Efth" in buoy_raw else "efth"
buoy_efth = buoy_raw[spec_var].sel(time=peak_time, method="nearest")

bulk_spec = mm.load_reconstruction_spectra(
    period, spectral=False, load=False, **recon_kw
)
part_spec = mm.load_reconstruction_spectra(
    period, spectral=True, load=False, **recon_kw
)
for label, ds in [("bulk", bulk_spec), ("partition", part_spec)]:
    if ds is None:
        raise FileNotFoundError(
            f"Missing {label} spectra cache for {period!r}. Run "
            f"mm.reconstruct(..., spectral={label == 'partition'}, recompute=True)."
        )

bulk_efth = bulk_spec["efth"].sel(sites=buoy_site, time=peak_time, method="nearest")
part_efth = part_spec["efth"].sel(sites=buoy_site, time=peak_time, method="nearest")

panels = [
    (buoy_efth, f"Buoy observed — {peak_time:%Y-%m-%d %H:%M}"),
    (bulk_efth, "Reconstruction bulk (partition −1)"),
    (part_efth, "Reconstruction spectral (partitions 0–3 sum)"),
]
images = [render_spec_image(da, title) for da, title in panels]
combined = stack_spec_images(images)

out = EXPORT_DIR / "fig06_hero_spectrum_storm_peak.png"
combined.save(out)
print(f"saved → {out}")

## Fig 7 — Metamodel pipeline (schematic)

Simple flow diagram for slides — no model data required.


In [ ]:
apply_slide_style()
steps = [
    "WHACS offshore\n(goals × partitions)",
    "MDA + LHS\n500 cases / goal",
    "SnapWave\nnearshore cases",
    "PCA + GPR\nmetamodel",
    "mm.reconstruct()\nhindcast",
    "Buoy / site\nvalidation",
]
fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
ax.set_xlim(0, len(steps))
ax.set_ylim(0, 1)
ax.axis("off")

for i, text in enumerate(steps):
    ax.text(
        i + 0.5,
        0.55,
        text,
        ha="center",
        va="center",
        fontsize=12,
        bbox=dict(
            boxstyle="round,pad=0.45",
            facecolor="#e8f4fc",
            edgecolor="#1f78b4",
            linewidth=1.2,
        ),
    )
    if i < len(steps) - 1:
        ax.annotate(
            "",
            xy=(i + 1.08, 0.55),
            xytext=(i + 0.92, 0.55),
            arrowprops=dict(arrowstyle="-|>", color="#444444", lw=1.8),
        )

ax.set_title("HyGloW — hybrid downscaling workflow", fontsize=16, pad=12)
save_fig(fig, "fig07_pipeline_schematic")

## Fig 8 — Input / output at one moment

**Top (output):** reconstructed spectra at the buoy for the same UTC hour.

**Bottom (input):** at each goal boundary snap, four coloured arrows show WHACS
partition forcing (``pdir`` + ``phs``) — wind sea + three swells.

Set ``forcing_moment`` in the config cell to pin a specific time; default is the
storm peak at the buoy's closest goal.


In [ ]:
apply_slide_style()
moment = resolve_forcing_moment()
print(f"Forcing moment: {moment}")

# --- output spectra (buoy) ---
buoy_raw = xr.open_dataset(resolve(buoy_nc))
spec_var = "Efth" if "Efth" in buoy_raw else "efth"
buoy_efth = buoy_raw[spec_var].sel(time=moment, method="nearest")

bulk_spec = mm.load_reconstruction_spectra(
    period, spectral=False, load=False, **recon_kw
)
part_spec = mm.load_reconstruction_spectra(
    period, spectral=True, load=False, **recon_kw
)
for label, ds in [("bulk", bulk_spec), ("partition", part_spec)]:
    if ds is None:
        raise FileNotFoundError(
            f"Missing {label} spectra cache for {period!r}. Run "
            f"mm.reconstruct(..., spectral={label == 'partition'}, recompute=True)."
        )

bulk_efth = bulk_spec["efth"].sel(sites=buoy_site, time=moment, method="nearest")
part_efth = part_spec["efth"].sel(sites=buoy_site, time=moment, method="nearest")

spec_panels = [
    (buoy_efth, "Buoy observed"),
    (bulk_efth, "Reconstruct bulk (p−1)"),
    (part_efth, "Reconstruct partitions 0–3"),
]

fig = plt.figure(figsize=(16.5, 12.0))
gs = gridspec.GridSpec(
    2,
    3,
    figure=fig,
    height_ratios=[1.2, 2.5],
    hspace=0.14,
    wspace=0.05,
)

for col, (da, title) in enumerate(spec_panels):
    ax_spec = fig.add_subplot(gs[0, col])
    img = render_spec_image(da, title, figsize=(6.0, 5.6), cmap="magma")
    ax_spec.imshow(np.asarray(img))
    ax_spec.axis("off")

ax_map = fig.add_subplot(gs[1, :])
cp.plot_base_map(ax=ax_map, title=None, show_topobathy=True)
draw_goal_partition_arrows(ax_map, moment)

buoy_meta = obs.get_coordinates()[buoy_instrument]
ax_map.plot(
    buoy_meta["lon"],
    buoy_meta["lat"],
    marker="s",
    color=cp.TYPE_COLORS["validation"],
    markersize=14,
    markeredgecolor="white",
    markeredgewidth=0.9,
    zorder=9,
)
ax_map.annotate(
    "validation buoy",
    (buoy_meta["lon"], buoy_meta["lat"]),
    xytext=(8, 8),
    textcoords="offset points",
    fontsize=11,
    color=cp.TYPE_COLORS["validation"],
    fontweight="bold",
    zorder=10,
    bbox=dict(
        boxstyle="round,pad=0.2", fc="white", ec=cp.TYPE_COLORS["validation"], alpha=0.9
    ),
)

x0, x1, y0, y1 = course_map_extent()
ax_map.set_xlim(x0, x1)
ax_map.set_ylim(y0, y1)
ax_map.set_aspect("equal", adjustable="box")

leg = ax_map.legend(
    handles=partition_legend_handles(),
    loc="lower left",
    fontsize=9.5,
    framealpha=0.96,
    edgecolor="#bbbbbb",
    title="WHACS partition forcing (input)",
    title_fontsize=10.5,
    ncol=1,
    borderpad=0.8,
)
leg.get_title().set_fontweight("bold")

fig.suptitle(
    f"INPUT → OUTPUT  |  {moment:%Y-%m-%d %H:%M} UTC  |  spectra at {buoy_site}",
    fontsize=16,
    fontweight="bold",
    y=0.995,
)
ax_map.set_title(
    "INPUT — partition arrows at each offshore goal  (direction = wave-from, length ∝ Hs)",
    fontsize=13,
    pad=10,
    color="#1a1a2e",
)

save_fig(fig, "fig08_input_output_partitions")

## Fig 9 — Offshore boundary conditions (concept)

Every SnapWave goal needs a directional wave spectrum imposed at its offshore
boundary — that's a structural requirement of this hybrid downscaling
approach, independent of *which* product supplies it. This figure shows the
storm-peak spectrum at each of the 16 boundary nodes without reference to a
specific source: a buoy, a different reanalysis, or WHACS (the next figure)
would all plug in here the same way.

It also shows the **reconstructed nearshore spectra** (~20–25 sites on the
20 m isobath) that result once that boundary condition is propagated inland —
still with no WHACS branding, to keep the point generic: *any* offshore
spectrum flows through the same downstream machinery.


In [ ]:
apply_slide_style()
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from utils.model.metamodel import SPECTRAL_DIR, SPECTRAL_FREQ
from wavespectra.construct import construct_partition

moment = resolve_forcing_moment()
print(f"Boundary spectra moment: {moment}")

active_goals = list(goals.keys())


def parametric_spectrum(hs: float, tp: float, dm: float, dspr: float):
    """Single-partition JONSWAP + cos^2s spectrum from bulk parameters."""
    return construct_partition(
        freq_name="jonswap",
        freq_kwargs={"freq": SPECTRAL_FREQ, "fp": 1.0 / tp, "hs": hs},
        dir_name="cartwright",
        dir_kwargs={"dir": SPECTRAL_DIR, "dm": dm, "dspr": dspr},
    )


def whacs_full_spectrum(time: pd.Timestamp, goal_id: int):
    """Sum of whatever partitions are available at one goal's boundary node.

    Named for the WHACS source used here, but the point of Fig 9 is that this
    node just needs *a* spectrum — Fig 10 puts the WHACS-specific label and
    partition arrows on top of the very same bubbles.
    """
    off_now = offshore_partitions_at(time).sel(goal=goal_id)
    parts = [
        parametric_spectrum(
            float(off_now[f"phs{p}"]),
            float(off_now[f"ptp{p}"]),
            float(off_now[f"pdir{p}"]),
            float(off_now[f"pspr{p}"]),
        )
        for p in range(4)
        if np.isfinite(float(off_now[f"phs{p}"])) and float(off_now[f"phs{p}"]) > 0
    ]
    return sum(parts)


def spec_bubble(
    da, *, figsize: tuple[float, float] = (1.8, 1.8), cmap: str = "magma"
) -> Image.Image:
    """Small transparent polar-spectrum raster for placing on the map."""
    fig = plt.figure(figsize=figsize)
    da.load().spec.plot(cmap=cmap, levels=16, normalised=True, add_colorbar=False)
    ax = plt.gca()
    ax.set_title("")
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(True, alpha=0.35, linewidth=0.5, color="white")
    buf = BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=DPI,
        bbox_inches="tight",
        pad_inches=0.02,
        transparent=True,
    )
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)


def pick_spaced_sites(
    lons: np.ndarray, lats: np.ndarray, min_spacing_deg: float
) -> np.ndarray:
    """Greedy longitude-ordered subsample keeping a minimum on-map spacing."""
    order = np.argsort(lons)
    picked: list[int] = []
    last = None
    for idx in order:
        pt = np.array([lons[idx], lats[idx]])
        if last is None or np.hypot(*(pt - last)) >= min_spacing_deg:
            picked.append(idx)
            last = pt
    return np.array(picked)


# --- nearshore sites: a ring on the 20 m isobath (metadata only, no bulk values used) ---
dyn = xr.open_dataset(resolve("outputs/SnapWave_Dynamic_202001.nc"))
isobath_mask = (dyn.site_source.values == "isobath") & (dyn.site_label.values == "20m")
ring_ids = np.asarray(dyn.sites.values)[isobath_mask]
ring_meta = dyn.isel(time=0).sel(sites=ring_ids)
keep = pick_spaced_sites(
    ring_meta.site_longitude.values, ring_meta.site_latitude.values, 0.16
)
nearshore_candidates = [str(s) for s in ring_meta.isel(sites=keep).sites.values]

template = mm.load_cases_goal(active_goals[0])
nearshore_site_ids = mm.resolve_reconstruction_sites(
    template, nearshore_candidates, goals=active_goals
)
n_dropped = len(nearshore_candidates) - len(nearshore_site_ids)
print(
    f"Nearshore spectra bubbles: {len(nearshore_site_ids)} "
    f"(of {len(ring_ids)} @ 20 m isobath, {n_dropped} dropped as not common to all goals)"
)

# --- real reconstruction, cached one calendar month at a time ---
_nearshore_month_cache: dict[str, xr.DataArray] = {}


def nearshore_efth_at(time: pd.Timestamp) -> xr.DataArray:
    """Reconstructed nearshore spectra at `time` (propagates the 4 offshore
    partitions through each goal's trained GP metamodel, then sums them with
    `wavespectra`). Computes/caches one month at a time via `mm.reconstruct`.
    """
    month = time.strftime("%Y-%m")
    if month not in _nearshore_month_cache:
        mm.reconstruct(
            month,
            n=mda_n,
            spectral=True,
            offshore_name=offshore_name,
            goals=None,
            wl=wl,
            sites=nearshore_site_ids,
            pca_variance=pca_variance,
            save_spectra=True,
        )
        spectra = mm.load_reconstruction_spectra(
            month,
            n=mda_n,
            spectral=True,
            offshore_name=offshore_name,
            goals=None,
            wl=wl,
            site_ids=nearshore_site_ids,
            pca_variance=pca_variance,
        )
        if spectra is None:
            raise FileNotFoundError(
                f"Missing nearshore spectra cache for {month!r} — check "
                "mm.reconstruct(save_spectra=True) above."
            )
        _nearshore_month_cache[month] = spectra["efth"]
    return _nearshore_month_cache[month].sel(time=time, method="nearest")


def plot_boundary_nearshore_figure(moment: pd.Timestamp, *, mode: str) -> plt.Figure:
    """Boundary + nearshore spectra bubbles at one moment.

    mode="concept" — generic styling: no WHACS wording, no partition arrows.
    mode="whacs"   — WHACS-branded styling with the 4 partition arrows overlaid.
    """
    show_whacs = mode == "whacs"
    nearshore_efth_now = nearshore_efth_at(moment)

    title = (
        "WHACS partitions — offshore forcing → nearshore reconstruction"
        if show_whacs
        else "Offshore boundary conditions — every goal needs a directional spectrum here"
    )

    fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
    cp.plot_base_map(ax=ax, title=title, show_topobathy=True)

    # nearshore: real reconstructed spectra (propagated + summed from the offshore partitions)
    for site_id in nearshore_site_ids:
        da = nearshore_efth_now.sel(sites=site_id)
        if not np.isfinite(da.values).any() or float(da.max()) <= 0:
            continue
        img = spec_bubble(da)
        lon = float(dyn.site_longitude.sel(sites=site_id))
        lat = float(dyn.site_latitude.sel(sites=site_id))
        ax.plot(
            lon,
            lat,
            "o",
            color="white",
            markeredgecolor="black",
            markersize=4,
            zorder=7,
        )
        ax.add_artist(
            AnnotationBbox(
                OffsetImage(np.asarray(img), zoom=0.15),
                (lon, lat),
                frameon=False,
                pad=0,
                zorder=6,
            )
        )

    # offshore boundary bubbles (+ WHACS partition arrows, in whacs mode)
    arrow_base_deg = 0.30
    edge_color = "#7a1f1f" if show_whacs else "#333333"
    for goal_key in sorted(goals, key=int):
        hc = goals[goal_key]["hindcast"]["whacs"]
        lon, lat = hc["longitude"], hc["latitude"]
        img = spec_bubble(whacs_full_spectrum(moment, int(goal_key)), figsize=(2.0, 2.0))
        ax.add_artist(
            AnnotationBbox(
                OffsetImage(np.asarray(img), zoom=0.20),
                (lon, lat),
                frameon=False,
                pad=0,
                zorder=8,
            )
        )
        ax.plot(
            lon, lat, "o", color="white", markeredgecolor=edge_color, markersize=5, zorder=9
        )

        if show_whacs:
            off_now = offshore_partitions_at(moment).sel(goal=int(goal_key))
            hs_vals = [float(off_now[f"phs{p}"]) for p in range(4)]
            hs_max_local = max([v for v in hs_vals if np.isfinite(v)] + [0.04])
            for p in range(4):
                phs = float(off_now[f"phs{p}"])
                pdir = float(off_now[f"pdir{p}"])
                if not np.isfinite(phs) or phs <= 0 or not np.isfinite(pdir):
                    continue
                color = PARTITION_STYLE[p][1]
                scale = arrow_base_deg * (0.45 + 0.55 * phs / hs_max_local)
                props, xytext, xy = _partition_arrowprops(
                    lon, lat, pdir, scale_deg=scale, color=color
                )
                ax.annotate("", xy=xy, xytext=xytext, arrowprops=props, zorder=10)

    legend_handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="white",
            markeredgecolor="black",
            markersize=8,
            linestyle="None",
            label=f"nearshore (reconstructed{' from WHACS' if show_whacs else ''}, 20 m isobath)",
        ),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="white",
            markeredgecolor=edge_color,
            markersize=8,
            linestyle="None",
            label=(
                "WHACS node (4 partitions summed)"
                if show_whacs
                else "boundary node — needs a directional spectrum (any source)"
            ),
        ),
    ]
    if show_whacs:
        for part, (label, color) in sorted(PARTITION_STYLE.items()):
            legend_handles.append(
                plt.Line2D(
                    [0],
                    [0],
                    color=color,
                    linewidth=3,
                    marker=">",
                    markersize=9,
                    markeredgewidth=0,
                    label=f"WHACS {label} arrow (dir = wave-from)",
                )
            )
    ax.legend(handles=legend_handles, loc="lower left", fontsize=8.5, framealpha=0.92)

    x0, x1, y0, y1 = course_map_extent()
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(f"{title} — {moment:%Y-%m-%d %H:%M} UTC")
    return fig


fig9 = plot_boundary_nearshore_figure(moment, mode="concept")
save_fig(fig9, "fig09_offshore_boundary_concept")


## Fig 10 — WHACS partitions: the forcing used in this case

Same boundary + nearshore bubbles as Fig 9 (same `moment`, same
`plot_boundary_nearshore_figure` helper), now with the **WHACS**-specific
branding turned on: WHACS decomposes offshore conditions into four partitions
per goal — wind sea + three swells (``phs``/``ptp``/``pdir``/``pspr`` 0–3).
The coloured arrows (direction = wave-from, length ∝ Hs) are drawn **on top
of** each boundary bubble so the summation into the WHACS spectrum can be
checked by eye, and the nearshore bubbles are the same reconstructed spectra
from Fig 9, now labelled as WHACS-driven.


In [ ]:
apply_slide_style()
fig10 = plot_boundary_nearshore_figure(moment, mode="whacs")
save_fig(fig10, "fig10_whacs_partitions_nearshore")


## Fig 9/10 sweep — sample moments across 2020

Renders both figures (concept + WHACS) at ~10 randomly sampled hours spread
across the whole of 2020, so a nicer or more illustrative storm moment than
the January storm peak above can be picked by eye.

Each distinct calendar month touched by the sample triggers one
`mm.reconstruct(spectral=True, ...)` run (~2–3 min each, cached under
`tmp/reconstruct/` for reruns) — budget several minutes to tens of minutes
depending on how many distinct months the random sample lands in. Outputs go
to `outputs/course_figures/sweep_2020/` and are **not** part of the numbered
Fig 1–10 set.


In [ ]:
apply_slide_style()
import time

N_SAMPLES = 10
SWEEP_SEED = 42

year_time_index = pd.to_datetime(open_offshore_waves(offshore_name).sel(time="2020").time.values)
rng = np.random.default_rng(SWEEP_SEED)
sweep_moments = sorted(
    pd.Timestamp(t) for t in rng.choice(year_time_index, size=N_SAMPLES, replace=False)
)
print(f"Sampled {len(sweep_moments)} moments across 2020 (seed={SWEEP_SEED}):")
for t in sweep_moments:
    print(f"  {t}")

sweep_dir = EXPORT_DIR / "sweep_2020"
sweep_dir.mkdir(parents=True, exist_ok=True)

t0 = time.time()
for i, t in enumerate(sweep_moments, start=1):
    print(f"\n[{i}/{len(sweep_moments)}] {t} (month {t:%Y-%m}) …")

    fig_c = plot_boundary_nearshore_figure(t, mode="concept")
    path_c = sweep_dir / f"fig09_concept_{t:%Y%m%dT%H}.png"
    fig_c.savefig(path_c, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig_c)
    print(f"  saved → {path_c}")

    fig_w = plot_boundary_nearshore_figure(t, mode="whacs")
    path_w = sweep_dir / f"fig10_whacs_{t:%Y%m%dT%H}.png"
    fig_w.savefig(path_w, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig_w)
    print(f"  saved → {path_w}")

print(f"\nDone — {len(sweep_moments)} pairs in {sweep_dir} ({time.time() - t0:.0f}s)")


## Export summary


In [ ]:
sorted(EXPORT_DIR.glob("fig*.png"))